In [1]:
import pandas as pd

home_team= pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\home_team.csv")  # مسیر هر فایلی که میخوای چک کنی

print("shape: ", home_team.shape)
print("\n--- data type ---")
print(home_team.dtypes)
print("\n--- null ---")
print(home_team.isnull().sum())
print("\n--- duplicate ---")
print(home_team.duplicated().sum())
print("\n---  first five  ---")
print(home_team.head())
print("\n--- ---")
print(home_team.describe())

shape:  (25610, 18)

--- data type ---
match_id           int64
name                 str
slug                 str
gender               str
user_count         int64
residence            str
birthplace           str
height           float64
weight           float64
plays                str
turned_pro       float64
current_prize    float64
total_prize      float64
player_id          int64
current_rank     float64
name_code            str
country              str
full_name            str
dtype: object

--- null ---
match_id             0
name                 0
slug                 0
gender              35
user_count           0
residence        18423
birthplace       10836
height           11287
weight           18578
plays            13126
turned_pro       20806
current_prize      130
total_prize         69
player_id            0
current_rank       273
name_code            0
country              9
full_name            0
dtype: int64

--- duplicate ---
7710

---  first five  ---
   match_i

In [2]:
away_team= pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\away_team.csv")  # مسیر هر فایلی که میخوای چک کنی

print("shape:", away_team.shape)
print("\n--- data type ---")
print(away_team.dtypes)
print("\n--- null ---")
print(away_team.isnull().sum())
print("\n--- duplicate ---")
print(away_team.duplicated().sum())
print("\n--- first five---")
print(away_team.head())
print("\n------")
print(away_team.describe())

shape: (24203, 18)

--- data type ---
match_id           int64
name                 str
slug                 str
gender               str
user_count         int64
residence            str
birthplace           str
height           float64
weight           float64
plays                str
turned_pro       float64
current_prize    float64
total_prize      float64
player_id          int64
current_rank     float64
name_code            str
country              str
full_name            str
dtype: object

--- null ---
match_id             0
name                 0
slug                 0
gender              33
user_count           0
residence        17185
birthplace       10308
height           10665
weight           17293
plays            12244
turned_pro       19476
current_prize      226
total_prize        133
player_id            0
current_rank       325
name_code            0
country              5
full_name            0
dtype: int64

--- duplicate ---
7322

--- first five---
   match_id   

**QUESTION ONE**

In [3]:
# فقط دو ستون مرتبط رو انتخاب می‌کنیم
all_players = pd.concat([home_team[['player_id', 'full_name']],
                         away_team[['player_id', 'full_name']]
], ignore_index=True)

# حالا drop_duplicates بر اساس player_id (نه کل ستون‌ها)
unique_players = all_players.drop_duplicates(subset='player_id')

print("Total number of players (unique):", unique_players['player_id'].nunique())

Total number of players (unique): 2644


**QUESTION TWO**

In [4]:
all_players = pd.concat([
    home_team[['player_id', 'full_name', 'height']],
    away_team[['player_id', 'full_name', 'height']]
], ignore_index=True)

unique_players = all_players.drop_duplicates(subset='player_id')

print("Total number of players (unique):", len(unique_players))
print("null:", unique_players['height'].isnull().sum())
print("Zeros:", (unique_players['height'] == 0).sum())
print()
#print("--- cleaned data ---")
#print(sorted(unique_players['height'].dropna().unique())[:30])   # 30 تای کوچیک‌تر
#print(sorted(unique_players['height'].dropna().unique())[-30:])  # 30 تای بزرگ‌تر
#print()
print(unique_players['height'].describe())

Total number of players (unique): 2644
null: 1327
Zeros: 0

count    1317.000000
mean        1.821374
std         0.080626
min         1.570000
25%         1.780000
50%         1.830000
75%         1.880000
max         2.080000
Name: height, dtype: float64


In [5]:
total_players = len(unique_players)
missing_height = unique_players['height'].isnull().sum()
valid_height_count = unique_players['height'].notnull().sum()

avg_height = unique_players['height'].mean()

print(f"total players: {total_players}")
print(f"playes with missing height values(null): {missing_height} ({missing_height/total_players*100:.1f}%)")
print(f"players with height values: {valid_height_count}")
print(f"average height(M): {avg_height:.2f} for {valid_height_count} players")

total players: 2644
playes with missing height values(null): 1327 (50.2%)
players with height values: 1317
average height(M): 1.82 for 1317 players


**QUESTION THREE**

In [ ]:
event_dfr = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\event.csv")
#CLEAN DATA FRAME AND DROPDUPLICATES
home_df = home_team.drop_duplicates().drop_duplicates(subset='match_id', keep='first')
away_df = away_team.drop_duplicates().drop_duplicates(subset='match_id', keep='first')
print("even data frame  raw: " , event_dfr.shape)
print("event data frames shape after cleaning: ", event_df.shape)
\
#KEEP THE COLS WE NEED
event_clean = event_df[['match_id', 'winner_code']].dropna(subset=['winner_code'])
home_clean = home_df[['match_id', 'player_id', 'full_name']]
away_clean = away_df[['match_id', 'player_id', 'full_name']]

even data frame  raw:  (35053, 10)
event data frames shape after cleaning:  (16873, 10)


In [7]:
merged = event_clean.merge(home_clean, on='match_id', how='inner')
merged = merged.merge(away_clean, on='match_id', how='inner', suffixes=('_home', '_away'))

print("shape afte rmerged:", merged.shape)


shape afte rmerged: (8828, 6)


In [ ]:
import numpy as np
#CHECK IF THE PLAYERS ARE HOME OR AWAY
merged['winner_id'] = np.where(merged['winner_code'] == 1, merged['player_id_home'], merged['player_id_away'])
merged['winner_name'] = np.where(merged['winner_code'] == 1, merged['full_name_home'], merged['full_name_away'])

#WIN COUNTS FOR REACH PLAYER
win_counts = merged.groupby(['winner_id', 'winner_name']).size().reset_index(name='wins')
win_counts = win_counts.sort_values('wins', ascending=False)

print("\n--- top ten players with the most win---")
print(win_counts.head(10))

top_player = win_counts.iloc[0]
print(f"\n final answer: {top_player['winner_name']} with {int(top_player['wins'])} have the highest win.")


--- top ten players with the most win---
      winner_id                        winner_name  wins
173       50901                      Popko, Dmitry    27
1115     231620                   Chidekh, Clement    22
364       82133  Dellien Velasco, Murkel Alejandro    20
895      202572                      Gengel, Marek    19
1100     230049              Jianu, Filip Cristian    18
847      197809                        Clarke, Jay    18
1653     341818                       Faria, Jaime    18
239       58515                  Collins, Danielle    18
235       58369                         Bolt, Alex    17
909      205282                  Kužmová, Katarína    17

 final answer: Popko, Dmitry with 27 have the highest win.


In [10]:
win_counts.head(10)

,winner_id,winner_name,wins
173,50901,"Popko, Dmitry",27
1115,231620,"Chidekh, Clement",22
364,82133,"Dellien Velasco, Murkel Alejandro",20
895,202572,"Gengel, Marek",19
1100,230049,"Jianu, Filip Cristian",18
847,197809,"Clarke, Jay",18
1653,341818,"Faria, Jaime",18
239,58515,"Collins, Danielle",18
235,58369,"Bolt, Alex",17
909,205282,"Kužmová, Katarína",17


**QUESTION FOUR**

In [12]:
time_df = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\time.csv")

print("time.shape:", time_df.shape)
print("\n cls", list(time_df.columns))
print("\n--- first five---")
print(time_df.head())
print("\n--- ---")
print(time_df.describe())

time.shape: (35671, 7)

 cls ['match_id', 'period_1', 'period_2', 'period_3', 'period_4', 'period_5', 'current_period_start_timestamp']

--- first five---
   match_id  period_1  period_2  period_3  period_4  period_5  \
0  11974053       NaN       NaN       NaN       NaN       NaN   
1  11974066       NaN       NaN       NaN       NaN       NaN   
2  11998445    3259.0    2639.0    4202.0       NaN       NaN   
3  11998446    2488.0    2375.0       NaN       NaN       NaN   
4  11998447    3741.0    1913.0       NaN       NaN       NaN   

   current_period_start_timestamp  
0                             NaN  
1                             NaN  
2                    1.706817e+09  
3                    1.706804e+09  
4                    1.706798e+09  

--- ---
           match_id       period_1       period_2       period_3  period_4  \
count  3.567100e+04   22576.000000   22471.000000    6886.000000       0.0   
mean   1.211936e+07    2968.423990    3191.026568    3292.027157       Na